# Lekse 12 - Reduksjon av chatthistorikk med Agent Scratchpad

Denne notatblokken demonstrerer hvordan man håndterer kontekst i lange samtaler ved bruk av Microsoft Agent Framework. Etter hvert som samtalene vokser, øker antallet tokens — til slutt overskrides modellens kontekstvindu. Vi tar tak i dette med et **mønster for kontekstsummering** og en **agent-scratchpad** for vedvarende minne.

## Hva du vil lære:
1. **Hvorfor kontekststyring er viktig**: Forståelse av tokenbegrensninger og kontekstvinduer
2. **Kontekstbevisste agenter**: Bygge agenter som håndterer sin egen samtalekontekst
3. **Mønster for kontekstsummering**: Bruke verktøy for å kondensere samtalehistorikk
4. **Agent Scratchpad**: Vedvarende minne som overlever kontekstreduksjon

## Forutsetninger:
- Azure OpenAI-oppsett med konfigurerte miljøvariabler
- Forståelse av grunnleggende agentkonsepter fra tidligere leksjoner


## Oppsett


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Hvorfor kontekststyring er viktig

Hver LLM har et begrenset **kontekstvindu** — maksimum antall tokens den kan behandle i en enkelt forespørsel. Etter hvert som en samtale med flere runder pågår:

- **Antallet tokens vokser lineært** med hver brukermelding og assistentsvar.
- **Prompt-tokens dominerer kostnaden** fordi hele historikken sendes på nytt hver runde.
- Til slutt **overskrides kontekstvinduet** og modellen enten trunkerer eller gir en feil.

### Strategier for å håndtere kontekst

| Strategi | Hvordan det fungerer | Kompromiss |
|---|---|---|
| **Trunkering** | Fjerner eldste meldinger | Mister tidlig kontekst |
| **Sammendrag** | Kondenserer eldre meldinger til et sammendrag | Noe informasjon går tapt, men hovedpunkter beholdes |
| **Notatblokk / Ekstern hukommelse** | Lagrer nøkkelfakta utenfor samtalen | Krever verktøy-kall, men overlever enhver reduksjon |

I denne notatboken kombinerer vi **sammendrag** med et **notatblokksverktøy** slik at agenten kan opprettholde kontinuitet selv når samtalehistorikken kondenseres.


## Lage en kontekstbevisst agent


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Simulering av en lang samtale

La oss gå gjennom en samtale med flere runder for å se hvordan kontekst bygger seg opp. Agenten skal beholde viktige detaljer (preferanser, budsjett, reisedatoer) gjennom rundene og vise kontinuitet.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Legg merke til hvordan agenten beholder konteksten fra tidligere runder — den vet om Japan, sushi, templer, fotografering, budsjettet på 3000 dollar, soloreise og tidsrammen i april. I en kort samtale fungerer dette bra, men etter hvert som samtalen vokser, blir det dyrt å sende hele historikken på nytt.

La oss fortsette samtalen med flere runder for å se akkumulering av kontekst:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Mønster for kontekstoppsummering

Etter hvert som samtalen vokser, kan vi bruke et **oppsummeringsverktøy** for å kondensere samlet kontekst til et kompakt format. Agenten kaller dette verktøyet for å registrere nøkkelpreferanser slik at selv om eldre meldinger blir fjernet, bevares den essensielle informasjonen.

Dette mønsteret er byggesteinen for mer sofistikert historikkreduksjon:
1. Agenten identifiserer nøkkelfakta fra samtalen
2. Den kaller oppsummeringsverktøyet for å lagre dem
3. Eldre meldinger kan trygt fjernes fordi oppsummeringen fanger det som betyr noe

Nedenfor definerer vi et `summarize_preferences`-verktøy som agenten kan kalle for å registrere en kompakt oppsummering av det den har lært.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Sammendrag

I denne leksjonen lærte du hvordan du håndterer kontekst i langvarige agent-samtaler ved hjelp av Microsoft Agent Framework:

### Nøkkelbegreper
- **Kontekstvinduer er begrensede** — hver token i samtalehistorikken koster penger og telles mot grensen.
- **Oppsummeringsverktøy** lar agenten kondensere akkumulert kontekst til kompakte sammendrag, noe som reduserer token-bruk samtidig som essensiell informasjon bevares.
- **Agentens notatblokker** gir vedvarende ekstern hukommelse som overlever enhver samtalereduksjon.

### Det du bygde
- En **kontekstbevisst agent** som opprettholder kontinuitet over samtaler med flere runder
- Et **oppsummeringsverktøy** (`summarize_preferences`) som registrerer viktige brukerdetaljer i et kompakt format
- En **samtale med flere runder** som demonstrerer kontekstopphold og håndtering av endringer

### Anvendelser i praksis
- **Kundeservice-boter**: Husker preferanser over lange supportsesjoner
- **Personlige assistenter**: Follower pågående prosjekter uten å måtte forklare kontekst på nytt
- **Utdanningstutorer**: Opprettholder elevens fremgang over mange interaksjoner

### Neste steg
- Implementer et fullstendig notatblokkverktøy med filbasert vedvarende lagring
- Legg til automatisk historikkforkortelse etter oppsummering
- Kombiner med vektordatabaser for semantisk hukommelsessøk
- Bygg agenter som kan fortsette samtaler dager senere med full kontekst


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi streber etter nøyaktighet, vær oppmerksom på at automatiske oversettelser kan inneholde feil eller unøyaktigheter. Det opprinnelige dokumentet på originalspråket skal betraktes som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som oppstår ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
